# 监控与可观测性 第4周:生产级实战

> **学习目标**:在 K8s 部署完整可观测性栈,设计 SLO Dashboard,编写 Runbook

---

## Day 22-23:K8s 部署 Prometheus Stack

```bash
# 添加 Helm repo
helm repo add prometheus-community https://prometheus-community.github.io/helm-charts

# 安装 kube-prometheus-stack
helm install monitoring prometheus-community/kube-prometheus-stack \
  --namespace monitoring --create-namespace \
  --set prometheus.prometheusSpec.retention=15d \
  --set grafana.adminPassword=CHANGE_ME

# 访问 Grafana
kubectl port-forward -n monitoring svc/monitoring-grafana 3000:80
```

一键部署:Prometheus + Grafana + Alertmanager + node-exporter + kube-state-metrics

## Day 24:SLO Dashboard 设计

四个核心面板:

```
┌────────────────────────────────────────────────────┐
│  SLI: 可用性  │  SLI: p99 延迟  │  Error Budget    │
│  99.97%       │  187ms          │  剩余 78%        │
├────────────────────────────────────────────────────┤
│  可用性趋势 (过去 7 天) — SLO: 99.9%              │
├────────────────────────────────────────────────────┤
│  错误预算燃尽图 (过去 30 天)                       │
├────────────────────────────────────────────────────┤
│  延迟分布 (Heatmap - Histogram 数据)               │
└────────────────────────────────────────────────────┘
```

## Day 25:K8s 日志采集

| 方案 | 部署方式 | 采集内容 | 特点 |
|------|---------|---------|------|
| Promtail DaemonSet | 每个 Node 一个 | Container stdout | Loki 原生,轻量 |
| Grafana Agent | DaemonSet | Metrics + Logs | 减少组件数 |
| OTEL Collector | DaemonSet/Deployment | Metrics + Logs + Traces | 厂商中立,CNCF 标准 |

## Day 26:编写 Runbook

每条告警都需要一个 Runbook:

```
# Runbook: HighErrorRate

## 第一步:确认影响范围 (1 min)
- 打开 Grafana Dashboard: 服务概览
- 确认是全部 endpoint 还是特定 endpoint 有问题

## 第二步:检查依赖 (2 min)
- 数据库: SELECT 1
- Redis: PING
- 外部 API: curl 关键依赖的 /health

## 第三步:查看日志 (3 min)
- Loki: {app="web"} |= "ERROR" (最近 10 分钟)
- 或 kubectl logs -l app=web --tail=100

## 第四步:查看 Trace (2 min)
- 从 Dashboard Exemplar 跳转到 Jaeger
- 找到错误 Trace,看具体哪个 Span 报错

## 第五步:尝试恢复 (5 min)
- 场景 A: DB 连接池耗尽 → 熔断 + 增加连接池
- 场景 B: 外部 API timeout → 降级或增加超时
- 场景 C: 代码 bug → 回滚到上一个版本
```

## Day 27:Dashboard as Code

```yaml
# /etc/grafana/provisioning/datasources/prometheus.yaml
apiVersion: 1
datasources:
  - name: Prometheus
    type: prometheus
    url: http://prometheus:9090
    access: proxy
    isDefault: true
```

把 Dashboard JSON 存在 Git 中。Grafana 通过 Provisioning 启动时自动加载。
Dashboard 变更走 PR review,像代码一样管理。

## Day 28:第4周综合练习

In [ ]:
print("""
生产级可观测性平台交付清单:

K8s 集群部署:
  ├── kube-prometheus-stack (Helm)
  ├── Loki + Grafana Agent (DaemonSet)
  ├── Jaeger (Helm)
  └── Python 应用 (多副本, 带完整 instrumentation)

Dashboard (JSON 提交到 Git):
  ├── SLO Dashboard (4 SLI + 错误预算燃尽)
  ├── App Dashboard (QPS / 延迟 / 错误 / 资源)
  └── Logs Dashboard (日志量 + 错误 Top N)

告警:
  ├── 5+ 条告警规则 (按 severity 分级)
  ├── Alertmanager 配置 (路由 + 分组 + 抑制)
  └── 3 条 Runbook

架构文档:
  ├── 组件拓扑图
  ├── 数据流向说明
  └── 运维操作手册

══════════════════════════════════
  监控与可观测性 4 周学习完成!
══════════════════════════════════

核心成果:能构建一个生产级可观测性平台,
将 MTTR(故障平均修复时间)从小时级缩短到分钟级。
""")